In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import col, to_timestamp, datediff, when

orders = spark.table("orders")

date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for c in date_cols:
    orders = orders.withColumn(c, to_timestamp(col(c)))

orders = orders.withColumn(
    "delivery_duration_days",
    datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
).withColumn(
    "on_time",
    when(col("order_delivered_customer_date") <= col("order_estimated_delivery_date"), 1).otherwise(0)
)

orders.write.mode("overwrite").format("delta").saveAsTable("orders_clean")

StatementMeta(, ed0b4013-2003-4f5a-b4ca-75669e5d4c91, 3, Finished, Available, Finished, False)

In [2]:
products = spark.table("products")
translation = spark.table("productcategorynametranslation")

products_clean = products.join(translation, on="product_category_name", how="left")
products_clean.write.mode("overwrite").format("delta").saveAsTable("products_clean")

StatementMeta(, ed0b4013-2003-4f5a-b4ca-75669e5d4c91, 4, Finished, Available, Finished, False)

In [3]:
from pyspark.sql.functions import avg

geo = spark.table("geolocation")
geo_clean = geo.groupBy("geolocation_zip_code_prefix").agg(
    avg("geolocation_lat").alias("lat"),
    avg("geolocation_lng").alias("lng")
)
geo_clean.write.mode("overwrite").format("delta").saveAsTable("geolocation_clean")

StatementMeta(, ed0b4013-2003-4f5a-b4ca-75669e5d4c91, 5, Finished, Available, Finished, False)

In [4]:
reviews = spark.table("orderreviews")
reviews_clean = reviews.filter(col("review_score").isNotNull())
reviews_clean.write.mode("overwrite").format("delta").saveAsTable("reviews_clean")

print(f"dropped {reviews.count() - reviews_clean.count()} rows with bad review_score")

StatementMeta(, ed0b4013-2003-4f5a-b4ca-75669e5d4c91, 6, Finished, Available, Finished, False)

dropped 2380 rows with bad review_score


In [5]:
from pyspark.sql.functions import explode, sequence, to_date, year, month, quarter, date_format

date_range = spark.sql("SELECT sequence(to_date('2016-01-01'), to_date('2018-12-31'), interval 1 day) as date")
dim_date = date_range.select(explode(col("date")).alias("Date")) \
    .withColumn("Year", year("Date")) \
    .withColumn("Month", month("Date")) \
    .withColumn("MonthName", date_format("Date", "MMMM")) \
    .withColumn("Quarter", quarter("Date"))

dim_date.write.mode("overwrite").format("delta").saveAsTable("dim_date")

StatementMeta(, ed0b4013-2003-4f5a-b4ca-75669e5d4c91, 7, Finished, Available, Finished, False)

In [6]:
for t in ["orders_clean", "products_clean", "geolocation_clean", "reviews_clean", "dim_date"]:
    print(t, spark.table(t).count())

StatementMeta(, ed0b4013-2003-4f5a-b4ca-75669e5d4c91, 8, Finished, Available, Finished, False)

orders_clean 99441
products_clean 32951
geolocation_clean 19015
reviews_clean 101782
dim_date 1096


In [4]:
from pyspark.sql.functions import to_date

orders_clean = spark.table("orders_clean")
orders_clean = orders_clean.withColumn("order_purchase_date", to_date("order_purchase_timestamp"))
orders_clean.write.mode("overwrite").format("delta").saveAsTable("orders_clean")

StatementMeta(, 405dff73-6cfc-477b-9b64-7a99a9312941, 6, Finished, Available, Finished, False)

In [2]:
orders_clean.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("orders_clean")

StatementMeta(, 405dff73-6cfc-477b-9b64-7a99a9312941, 4, Finished, Available, Finished, False)

In [5]:
spark.table("orders_clean").printSchema()

StatementMeta(, 405dff73-6cfc-477b-9b64-7a99a9312941, 7, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- delivery_duration_days: integer (nullable = true)
 |-- on_time: integer (nullable = true)
 |-- order_purchase_date: date (nullable = true)



In [1]:
from pyspark.sql.functions import col

reviews_clean = spark.table("reviews_clean")
reviews_clean = reviews_clean.withColumn("review_score", col("review_score").cast("int"))
reviews_clean.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("reviews_clean")

StatementMeta(, 1e4afe6c-f53b-4e83-bd5d-ee986d2ea424, 3, Finished, Available, Finished, False)

In [2]:
spark.table("reviews_clean").printSchema()

StatementMeta(, 1e4afe6c-f53b-4e83-bd5d-ee986d2ea424, 4, Finished, Available, Finished, False)

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)



In [3]:
from pyspark.sql.functions import col
bad = spark.table("reviews_clean").filter(col("review_score").isNull()).count()
print(f"{bad} rows have null review_score after cast")

StatementMeta(, 1e4afe6c-f53b-4e83-bd5d-ee986d2ea424, 5, Finished, Available, Finished, False)

2555 rows have null review_score after cast


In [4]:
from pyspark.sql.functions import col

reviews_clean = spark.table("reviews_clean")
reviews_clean = reviews_clean.withColumn("review_score", col("review_score").cast("int"))
reviews_clean = reviews_clean.filter(col("review_score").isNotNull())
reviews_clean.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("reviews_clean")

print(f"reviews_clean now has {reviews_clean.count()} rows")

StatementMeta(, 1e4afe6c-f53b-4e83-bd5d-ee986d2ea424, 6, Finished, Available, Finished, False)

reviews_clean now has 99227 rows
